<a href="https://colab.research.google.com/github/A8stern/PyAD_mobile_2025/blob/main/lab_6_kovalev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа №6

**Выполнил: Ковалев Глеб К3341 367291**


## Описание датасета
Для анализа выбран датасет **`bmw.csv`** — таблица с объявлениями и характеристиками **подержанных автомобилей BMW**. Данные содержат основные технические параметры автомобилей, а также ценовые показатели, используемые для анализа рынка подержанных авто.

**Источник:**: Kaggle, "BMW Cars": https://www.kaggle.com/datasets/thedrzee/bmw-carsdataset

---

## Состав набора данных (поля таблицы)

- **model** — модель или серия автомобиля BMW (например, *3 Series*, *X5* и т.п.).  
- **year** — год выпуска автомобиля.  
- **price** — цена автомобиля (валюта в датасете не указана; в подобных наборах данных обычно используется цена из объявлений).  
- **transmission** — тип коробки передач (*Manual*, *Automatic*, *Semi-Auto*).  
- **mileage** — пробег автомобиля (единицы измерения не указаны; в подобных датасетах часто используется пробег в милях).  
- **fuelType** — тип топлива (*Petrol*, *Diesel*, *Hybrid*, *Electric*, *Other*).  
- **tax** — транспортный налог (числовое значение; единицы зависят от страны и источника данных).  
- **mpg** — показатель экономичности топлива (miles per gallon); в данных присутствуют выбросы.  
- **engineSize** — объём двигателя автомобиля (как правило, в литрах).

## 0. Загрузка данных

In [72]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy as scp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import PowerTransformer, RobustScaler

In [73]:
data = pd.read_csv('bmw.csv')

In [74]:
data.head()

,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize
0,5 Series,2014,11200,Automatic,67068,Diesel,125,57.6,2.0
1,6 Series,2018,27000,Automatic,14827,Petrol,145,42.8,2.0
2,5 Series,2016,16000,Automatic,62794,Diesel,160,51.4,3.0
3,1 Series,2017,12750,Automatic,26676,Diesel,145,72.4,1.5
4,7 Series,2014,14500,Automatic,39554,Diesel,160,50.4,3.0


## 1. Анализ исходных данных

In [75]:
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = data.select_dtypes(include=['object']).columns.tolist()

print("Числовые признаки:", numeric_cols)
print("Категориальные признаки:", categorical_cols)

def numeric_stats_table(df: pd.DataFrame, cols, exclude=None, cv_as_percent=True):
    exclude = set(exclude or [])
    rows = []

    for col in cols:
        if col in exclude:
            continue

        s = df[col]
        s_clean = s.dropna()

        modes = s_clean.mode()
        mode_str = ", ".join(map(str, modes.tolist())) if len(modes) else np.nan

        mean_val = s_clean.mean()
        std_val = s_clean.std(ddof=1)

        if pd.isna(mean_val) or np.isclose(mean_val, 0.0):
            cv = np.nan
        else:
            cv = std_val / mean_val
            if cv_as_percent:
                cv *= 100

        rows.append({
            "feature": col,
            "count": int(s_clean.shape[0]),
            "missing": int(s.isna().sum()),
            "mean": mean_val,
            "median": s_clean.median(),
            "mode": mode_str,
            "std": std_val,
            "min": s_clean.min(),
            "max": s_clean.max(),
            "skew": scp.stats.skew(s_clean, bias=False),
            "excess_kurtosis": scp.stats.kurtosis(s_clean, fisher=True, bias=False),
            "cv_%": cv if cv_as_percent else (std_val / mean_val if not np.isclose(mean_val, 0.0) else np.nan),
        })

    out = pd.DataFrame(rows).set_index("feature")

    float_cols = ["mean","median","std","min","max","skew","excess_kurtosis","cv_%"]
    out[float_cols] = out[float_cols].astype(float).round(4)
    return out

stats_df = numeric_stats_table(data, numeric_cols, cv_as_percent=True)

print("\n" + "="*60)
print("СТАТИСТИКИ ДЛЯ ЧИСЛОВЫХ ПРИЗНАКОВ (mean/median/mode/std/skew/kurtosis/CV)")
print("="*60)
print(stats_df)

Числовые признаки: ['year', 'price', 'mileage', 'tax', 'mpg', 'engineSize']
Категориальные признаки: ['model', 'transmission', 'fuelType']

СТАТИСТИКИ ДЛЯ ЧИСЛОВЫХ ПРИЗНАКОВ (mean/median/mode/std/skew/kurtosis/CV)
            count  missing        mean   median   mode         std     min  \
feature                                                                      
year        10781        0   2017.0789   2017.0   2019      2.3490  1996.0   
price       10781        0  22733.4089  20462.0  20980  11415.5282  1200.0   
mileage     10781        0  25496.9866  18347.0    123  25143.1926     1.0   
tax         10781        0    131.7021    145.0    145     61.5108     0.0   
mpg         10781        0     56.3990     53.3   65.7     31.3370     5.5   
engineSize  10781        0      2.1678      2.0    2.0      0.5521     0.0   

                 max    skew  excess_kurtosis     cv_%  
feature                                                 
year          2020.0 -1.7892           7.1655  

## Насколько распределение отличается от нормального?

Для нормального распределения характерны:
- коэффициент асимметрии = 0;
- нормальный эксцесс = 3.

По полученным результатам **ни один из числовых признаков не следует нормальному распределению**.

Наиболее сильное отклонение от нормальности наблюдается у признака **mpg**, где одновременно присутствуют крайне высокая асимметрия и эксцесс.

---

## Есть ли признаки с выраженной асимметрией (> 1 или < –1)?

Выраженная асимметрия (|skew| > 1) наблюдается у следующих признаков:

- **year**: skew = –1.7892 — сильная левосторонняя асимметрия  
- **price**: skew = 1.5869 — правосторонняя асимметрия  
- **mileage**: skew = 1.3831 — правосторонняя асимметрия  
- **mpg**: skew = 9.6292 — крайне сильная правосторонняя асимметрия  

Признаки **tax** (0.8278) и **engineSize** (0.8515) имеют умеренную асимметрию, но не превышают порог |skew| > 1.

---

## Какие признаки имеют высокий эксцесс (тяжёлые хвосты)?

Если считать высоким эксцесс значений **больше 3**, то тяжёлые хвосты наблюдаются у:

- **mpg**: excess kurtosis = 120.4593 — экстремально тяжёлые хвосты  
- **tax**: excess kurtosis = 9.2587  
- **year**: excess kurtosis = 7.1655  
- **price**: excess kurtosis = 4.0215  

Умеренно пониженный эксцесс (чуть ниже нормального распределения):

- **engineSize**: 2.4386  
- **mileage**: 2.2260  

---

## Какие признаки наиболее и наименее вариабельны?

Для оценки вариабельности используется **коэффициент вариации (CV, %)**.

### Наиболее вариабельные признаки:
- **mileage** — 98.6124%  
- **mpg** — 55.5629%  
- **price** — 50.2148%  
- **tax** — 46.7045%  

### Наименее вариабельные признаки:
- **year** — 0.1165% (минимальная относительная вариативность)  
- **engineSize** — 25.4665%  

Таким образом, признаки **mileage**, **mpg** и **price** характеризуются наибольшим относительным разбросом значений, тогда как **year** является наиболее стабильным признаком.

## 2. Визуализация Plotly

In [76]:
def plot_numeric_distributions_from_stats(data, stats_df):
    cols = stats_df.index.tolist()
    n = len(cols)

    fig = make_subplots(
        rows=n,
        cols=2,
        column_widths=[0.7, 0.3],
        horizontal_spacing=0.08,
        subplot_titles=[
            title
            for col in cols
            for title in (f"{col}: Histogram + KDE", f"{col}: Boxplot")
        ]
    )

    for i, col in enumerate(cols):
        s = data[col].dropna()

        sk = stats_df.loc[col, "skew"]
        kt = stats_df.loc[col, "excess_kurtosis"]
        cv = stats_df.loc[col, "cv_%"]

        annotation_text = (
            f"Skew: {sk:.2f}<br>"
            f"Kurtosis: {kt:.2f}<br>"
            f"CV: {cv:.2f}%"
        )

        fig.add_trace(
            go.Histogram(
                x=s,
                nbinsx=40,
                histnorm="probability density",
                showlegend=False
            ),
            row=i + 1,
            col=1
        )

        kde = scp.stats.gaussian_kde(s)
        x_vals = np.linspace(s.min(), s.max(), 300)

        fig.add_trace(
            go.Scatter(
                x=x_vals,
                y=kde(x_vals),
                mode="lines",
                showlegend=False
            ),
            row=i + 1,
            col=1
        )

        fig.add_trace(
            go.Box(
                x=s,
                showlegend=False
            ),
            row=i + 1,
            col=2
        )

        fig.add_annotation(
            x=0.98,
            y=1 - i / n - 0.035,
            xref="paper",
            yref="paper",
            text=annotation_text,
            showarrow=False,
            align="right",
            font=dict(size=11)
        )

    fig.update_layout(
        height=300 * n,
        title="Распределения числовых признаков (Histogram + KDE, Boxplot)",
        template="plotly_white"
    )

    fig.show()

plot_numeric_distributions_from_stats(data, stats_df)

## 3. Нормализация / трансформация

In [77]:
def compute_metrics(x: pd.Series) -> dict:
    x = pd.Series(x).dropna().astype(float)
    mean_val = x.mean()
    std_val = x.std(ddof=1)
    cv = (std_val / mean_val) * 100 if (not np.isclose(mean_val, 0.0) and not np.isnan(mean_val)) else np.nan

    return {
        "skew": scp.stats.skew(x, bias=False),
        "excess_kurtosis": scp.stats.kurtosis(x, fisher=True, bias=False),
        "cv_%": cv
    }

def apply_transforms(x: pd.Series) -> dict:
    x = pd.Series(x).dropna().astype(float)
    out = {"original": x}

    if (x >= 0).all():
        out["log1p"] = np.log1p(x)
        out["sqrt"] = np.sqrt(x)

    out = {k: v for k, v in out.items() if v is not None}
    return out

In [78]:
features = ["mileage", "mpg", "year"]

results = []

for col in features:
    trans = apply_transforms(data[col])

    for tname, series in trans.items():
        m = compute_metrics(series)
        results.append({
            "feature": col,
            "transform": tname,
            "skew": round(m["skew"], 4),
            "excess_kurtosis": round(m["excess_kurtosis"], 4),
            "cv_%": round(m["cv_%"], 4)
        })

metrics_df = pd.DataFrame(results).set_index(["feature", "transform"])
print(metrics_df)

                     skew  excess_kurtosis     cv_%
feature transform                                  
mileage original   1.3831           2.2260  98.6124
        log1p     -1.8353           3.4981  21.3863
        sqrt       0.2833          -0.5961  58.2886
mpg     original   9.6292         120.4593  55.5629
        log1p      1.4060          12.4921   7.6352
        sqrt       4.9163          45.3871  18.7945
year    original  -1.7892           7.1655   0.1165
        log1p     -1.7996           7.2529   0.0153
        sqrt      -1.7944           7.2091   0.0583


In [79]:
def plot_before_after(data: pd.DataFrame, col_name: str):
    x = data[col_name].dropna().astype(float)

    trans_dict = {"original": x}

    if (x >= 0).all():
        trans_dict["log1p"] = np.log1p(x)
        trans_dict["sqrt"] = np.sqrt(x)
    else:
        print(f"[WARN] '{col_name}' содержит отрицательные значения → log1p/sqrt пропущены")

    show = list(trans_dict.keys())
    n = len(show)

    fig = make_subplots(
        rows=n, cols=2,
        column_widths=[0.7, 0.3],
        horizontal_spacing=0.08,
        subplot_titles=[
            title
            for t in show
            for title in (f"{col_name} — {t}: Hist + KDE", f"{col_name} — {t}: Box")
        ]
    )

    for i, tname in enumerate(show):
        s = trans_dict[tname].dropna().astype(float)

        m = compute_metrics(s)
        annotation_text = (
            f"Skew: {m['skew']:.2f}<br>"
            f"Kurtosis: {m['excess_kurtosis']:.2f}<br>"
            f"CV: {m['cv_%']:.2f}%"
        )

        fig.add_trace(
            go.Histogram(
                x=s,
                nbinsx=40,
                histnorm="probability density",
                showlegend=False
            ),
            row=i+1, col=1
        )

        kde = scp.stats.gaussian_kde(s)
        x_vals = np.linspace(s.min(), s.max(), 300)

        fig.add_trace(
            go.Scatter(
                x=x_vals,
                y=kde(x_vals),
                mode="lines",
                showlegend=False
            ),
            row=i+1, col=1
        )

        fig.add_trace(
            go.Box(
                x=s,
                showlegend=False
            ),
            row=i+1, col=2
        )

        fig.add_annotation(
            x=0.98,
            y=1 - i / n - 0.04,
            xref="paper",
            yref="paper",
            text=annotation_text,
            showarrow=False,
            align="right",
            font=dict(size=11)
        )

    fig.update_layout(
        height=280 * n,
        title=f"Сравнение распределений: {col_name} (до/после трансформаций)",
        template="plotly_white"
    )

    fig.show()

In [80]:
for col in ["mileage", "mpg", "year"]:
    plot_before_after(data, col)

## 4. Вывод

### 1) Какие трансформации оказались наиболее эффективными и почему?

**mileage**
- **До:** Skew = 1.38, Kurtosis = 2.23, CV = 98.61% → сильная правосторонняя асимметрия и очень высокая вариативность.
- **log1p:** Skew = -1.84, Kurtosis = 3.50, CV = 21.39%  
  - **сильно снизился CV** (разброс относительно среднего стал намного меньше), но распределение стало **лево-асимметричным** и эксцесс вырос (хвосты остались тяжёлыми).
- **sqrt:** Skew = 0.28, Kurtosis = -0.60, CV = 58.29%  
  - **лучше всего “выпрямило” форму распределения** (асимметрия близка к 0, эксцесс стал около 0 и даже отрицательным), но **CV снизился не так сильно**, как у log1p.

**Итог по mileage:**  
- если цель — **сделать распределение ближе к нормальному**, то наиболее эффективна **sqrt**;  
- если цель — **снизить относительный разброс (CV)**, то наиболее эффективна **log1p**.

---

**mpg**
- **До:** Skew = 9.63, Kurtosis = 120.46, CV = 55.56% → крайне сильная асимметрия и экстремальный эксцесс (очень тяжёлые хвосты/выбросы).
- **log1p:** Skew = 1.41, Kurtosis = 12.49, CV = 7.64%  
  - **самое сильное улучшение по всем метрикам**: резкое снижение асимметрии, эксцесса и CV.  
- **sqrt:** Skew = 4.92, Kurtosis = 45.39, CV = 18.79%  
  - улучшение есть, но распределение остаётся сильно асимметричным и с тяжёлыми хвостами.

**Итог по mpg:**  
Наиболее эффективная трансформация — **log1p**, т.к. она заметно снижает и асимметрию, и “тяжесть хвостов”, и относительную вариативность.

---

**year**
- **До:** Skew = -1.79, Kurtosis = 7.17, CV = 0.12%  
- **log1p / sqrt:** Skew и Kurtosis практически не меняются (остаются около -1.8 и 7.2), CV становится ещё меньше.

**Итог по year:**  
Трансформации **не дали существенного улучшения формы распределения**. Это логично, потому что `year` — дискретный и ограниченный по диапазону признак, и “нормализовать” его логарифмом/корнем не очень осмысленно.

---

### 2) Стоит ли применять эти преобразования перед обучением ML-моделей?

**Да, но выбор зависит от типа модели.**

  **Когда применять полезно (рекомендуется):**
- Для моделей, чувствительных к масштабу и распределению признаков:
  - линейная/логистическая регрессия,
  - kNN,
  - SVM,
  - нейросети,
  - методы, использующие расстояния/градиенты.
- Для таких моделей сильная асимметрия и тяжёлые хвосты ухудшают обучение и качество, поэтому:
  - `mpg` **лучше трансформировать (log1p)** — это существенно снижает влияние выбросов и делает распределение “ровнее”.
  - `mileage` можно трансформировать:
    - **sqrt**, если важна “похожесть на нормальное”,
    - **log1p**, если важнее стабилизировать разброс (снизить CV).

  **Когда это не обязательно:**
- Для деревьев решений и ансамблей (RandomForest, XGBoost/LightGBM/CatBoost):  
  они менее чувствительны к нормальности распределений, и трансформации чаще дают небольшой эффект.
  Однако при экстремальных выбросах (как у `mpg`) лог-трансформация всё равно может помочь.

  **Про `year`:**
- Лучше оставить `year` как есть (или рассмотреть как категориальный/порядковый признак),  
  потому что трансформации не улучшают skew/kurtosis и практически ничего не дают модели.